# LeetCode 720 — Longest Word in Dictionary bằng Python

Link đề: https://leetcode.com/problems/longest-word-in-dictionary/ · Độ khó: **Medium** · Chủ đề: Trie, Array, Hash Table, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def longestWord(self, words: List[str]) -> str:
        ...
```

Ghi chú: một từ được coi là "xây dựng được" nếu **mọi tiền tố** của nó (độ dài `1, 2, ..., len - 1`) đều là từ trong `words`. Notebook đi từ vét cạn, qua tập băm, sắp xếp + tập các từ đã xây, tới Trie (hai cách: tra từng từ, và DFS trên cây).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm từ **dài nhất** trong `words` mà **mọi tiền tố** của nó cũng nằm trong `words`; nếu hòa độ dài thì chọn từ **nhỏ nhất theo thứ tự từ điển**; không có thì trả `""`.
2. Nắm các công cụ Python: cắt tiền tố `w[:k]`, `set` tra cứu `O(1)`, `sorted(..., key=...)` với khóa nhiều thành phần `(len(w), w)`, so sánh chuỗi theo từ điển, class `Trie` với `children` và `is_end`.
3. Đi từ **kiểm tra từng tiền tố trong list** → **kiểm tra từng tiền tố trong set** → **sắp xếp rồi chỉ nhìn "từ cha" `w[:-1]`** → **Trie: tra từng từ** → **Trie: DFS chỉ đi qua nút `is_end`**.
4. Hiểu mấu chốt: "xây dựng được" có tính **đệ quy**: `w` xây được khi `w[:-1]` xây được và `w` có trong từ điển. Nếu xử lý từ **ngắn trước dài sau**, chỉ cần tra một lần "từ cha".
5. Tránh các lỗi kinh điển: chỉ kiểm tra từ cha có **trong `words`** mà không kiểm tra cha đó **xây được**, quên xử lý hòa (độ dài bằng nhau → từ điển nhỏ hơn), quên trường hợp trả `""`, và nhầm "tiền tố" với "chuỗi con".

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` là một từ điển tiếng Anh. Hãy trả về **từ dài nhất** trong `words` có thể được xây dựng **từng ký tự một** bằng các từ khác trong `words`.

Nếu có nhiều đáp án, trả về từ **dài nhất** có **thứ tự từ điển nhỏ nhất**. Nếu không có đáp án, trả về chuỗi rỗng `""`.

Lưu ý: từ được xây từ trái sang phải, mỗi lần thêm một ký tự vào **cuối** một từ trước đó.

**Ví dụ 1**

- Input: `words = ["w","wo","wor","worl","world"]`
- Output: `"world"`
- Giải thích: `"world"` được xây bằng `"w"`, `"wo"`, `"wor"`, `"worl"`.

**Ví dụ 2**

- Input: `words = ["a","banana","app","appl","ap","apply","apple"]`
- Output: `"apple"`
- Giải thích: cả `"apply"` và `"apple"` đều xây được, nhưng `"apple"` nhỏ hơn `"apply"` theo từ điển.

**Ràng buộc:**

- `1 <= words.length <= 1000`
- `1 <= words[i].length <= 30`
- `words[i]` chỉ gồm chữ thường tiếng Anh.

Chú ý: ràng buộc không cấm các từ **trùng nhau**.

In [1]:
from typing import List
import random

examples = [
    (["w", "wo", "wor", "worl", "world"], "world"),
    (["a", "banana", "app", "appl", "ap", "apply", "apple"], "apple"),
]
for words, expected in examples:
    print(words, "->", repr(expected))

['w', 'wo', 'wor', 'worl', 'world'] -> 'world'
['a', 'banana', 'app', 'appl', 'ap', 'apply', 'apple'] -> 'apple'


## 3. Kiến thức nền

### 3.1. Tiền tố và cắt chuỗi

Tiền tố (prefix) là đoạn **bắt đầu từ ký tự đầu**. `w[:k]` là tiền tố độ dài `k`; `w[:-1]` là "từ cha" (bỏ ký tự cuối). Mọi tiền tố của `"apple"`: `a, ap, app, appl, apple`.

In [2]:
w = "apple"
print([w[:k] for k in range(1, len(w) + 1)])
print(repr(w[:-1]), repr("a"[:-1]))          # từ 1 ký tự có "cha" là chuỗi rỗng
print([w[:k] for k in range(1, len(w))])     # tiền tố thực sự (không gồm chính w)

['a', 'ap', 'app', 'appl', 'apple']
'appl' ''
['a', 'ap', 'app', 'appl']


### 3.2. `set` và tra cứu nhanh

`x in list` phải duyệt cả list (`O(n)`); `x in set` trung bình `O(1)`. Với `n = 1000` từ và hàng chục nghìn lần hỏi, đổi list sang set tiết kiệm đáng kể. Set cũng tự loại phần tử trùng.

In [3]:
words = ["a", "ap", "app", "app"]
as_set = set(words)
print(as_set, len(as_set))
print("ap" in words, "ap" in as_set)
print("" in as_set, "" in {""})

{'a', 'ap', 'app'} 3
True True
False True


### 3.3. So sánh chuỗi theo từ điển và `sorted` với khóa tuple

Python so sánh chuỗi theo **từ điển** (từng ký tự từ trái sang phải; chuỗi ngắn là tiền tố của chuỗi dài thì nhỏ hơn): `"apple" < "apply"` (vì `e < y`), `"ap" < "app"`. `sorted(words)` sắp theo từ điển. `sorted(words, key=lambda w: (len(w), w))` sắp theo **độ dài tăng dần**, hòa thì theo từ điển — tuple được so sánh phần tử đầu rồi tới phần tử sau.

In [4]:
print("apple" < "apply", "ap" < "app", "b" < "ab")
words = ["banana", "a", "apply", "app", "apple", "ap", "appl"]
print(sorted(words))
print(sorted(words, key=lambda w: (len(w), w)))

True True False
['a', 'ap', 'app', 'appl', 'apple', 'apply', 'banana']
['a', 'ap', 'app', 'appl', 'apple', 'apply', 'banana']


### 3.4. Trie (cây tiền tố)

**Trie** là cây mà mỗi cạnh là một ký tự, đường đi từ gốc tới một nút là một tiền tố; cờ `is_end` cho biết "tới đây là một từ hoàn chỉnh trong từ điển". Các từ chung tiền tố dùng chung đường đi. Bài này hỏi về "mọi tiền tố đều là từ" → trên Trie nghĩa là **mọi nút trên đường đi tới `w` đều có `is_end = True`**.

In [5]:
class Trie:
    def __init__(self):
        self.children = {}        # ký tự -> Trie
        self.is_end = False

    def insert(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = Trie()
            node = node.children[ch]
        node.is_end = True

t = Trie()
for w in ["a", "ap", "app", "banana"]:
    t.insert(w)
print(sorted(t.children))
print("a.is_end:", t.children["a"].is_end, "| a->p.is_end:", t.children["a"].children["p"].is_end)
node = t.children["b"]
print("b.is_end:", node.is_end, "(b chưa là từ, dù 'banana' đi qua)")

['a', 'b']
a.is_end: True | a->p.is_end: True
b.is_end: False (b chưa là từ, dù 'banana' đi qua)


### 3.5. `max` với khóa và cách chọn "dài nhất, hòa thì nhỏ nhất theo từ điển"

Muốn "dài nhất, hòa thì nhỏ nhất theo từ điển" ta có hai cách: (a) cập nhật `ans` khi `len(w) > len(ans)` **hoặc** (`len(w) == len(ans)` và `w < ans`); (b) duyệt các từ theo thứ tự `(len, w)` tăng dần và chỉ cập nhật khi **dài hơn hẳn** (từ đầu tiên của mỗi độ dài đã là nhỏ nhất).

In [6]:
cands = ["apply", "apple", "ant"]
best = ""
for w in cands:
    if len(w) > len(best) or (len(w) == len(best) and w < best):
        best = w
print(best)
print(min(cands, key=lambda w: (-len(w), w)))     # cùng kết quả bằng khóa (-độ dài, từ điển)

apple
apple


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `words = ["a","banana","app","appl","ap","apply","apple"]`.

- `a`: một ký tự → xây được (cha là rỗng).
- `ap`: cha `a` có trong từ điển → xây được.
- `app`: cha `ap` → được. `appl`: cha `app` → được.
- `apple`, `apply`: cha `appl` → được; độ dài `5`.
- `banana`: các tiền tố `b`, `ba`, `ban`, ... **không** có trong từ điển → **không** xây được (dù dài 6).

Các từ xây được dài nhất: `apple`, `apply` (độ dài 5) → chọn `apple` vì nhỏ hơn theo từ điển.

**Quan sát quan trọng:** từ `w` xây được ⇔ **cha `w[:-1]` xây được** *và* `w ∈ words` (hoặc `w` có đúng 1 ký tự). Nếu ta xét các từ theo thứ tự **từ ngắn đến dài**, khi tới `w` mọi từ ngắn hơn đã được phân loại, nên chỉ cần tra cha trong "tập các từ đã xây được".

In [7]:
def explain(words):
    words_set = set(words)
    for w in sorted(set(words), key=lambda x: (len(x), x)):
        missing = [w[:k] for k in range(1, len(w)) if w[:k] not in words_set]
        print(f"{w!r:9} tiền tố thiếu: {missing if missing else 'không'} -> {'xây được' if not missing else 'KHÔNG'}")

explain(["a", "banana", "app", "appl", "ap", "apply", "apple"])

'a'       tiền tố thiếu: không -> xây được
'ap'      tiền tố thiếu: không -> xây được
'app'     tiền tố thiếu: không -> xây được
'appl'    tiền tố thiếu: không -> xây được
'apple'   tiền tố thiếu: không -> xây được
'apply'   tiền tố thiếu: không -> xây được
'banana'  tiền tố thiếu: ['b', 'ba', 'ban', 'bana', 'banan'] -> KHÔNG


## 5. Cách 1 — Brute Force: kiểm tra mọi tiền tố trong **list**

### Ý tưởng

Với mỗi từ `w`, với mỗi `k = 1 .. len(w) - 1`, kiểm tra `w[:k] in words` (duyệt list). Nếu mọi tiền tố đều có, `w` là ứng viên; cập nhật đáp án theo quy tắc dài nhất/từ điển nhỏ nhất.

### Code

In [8]:
def longest_brute(words):
    ans = ""
    for w in words:
        ok = True
        for k in range(1, len(w)):
            if w[:k] not in words:          # tra trong LIST: O(n)
                ok = False
                break
        if ok and (len(w) > len(ans) or (len(w) == len(ans) and w < ans)):
            ans = w
    return ans

for words, expected in examples:
    got = longest_brute(words)
    print(repr(got))
    assert got == expected
print(repr(longest_brute(["b", "ab"])), repr(longest_brute(["x"])))

'world'
'apple'
'b' 'x'


### Độ phức tạp và nút thắt

- Gọi `n` là số từ, `L` độ dài tối đa. Mỗi từ có `<= L` tiền tố, mỗi lần tra list tốn `O(n)` và so sánh chuỗi `O(L)`.
- Time `O(n * L * n * L) = O(n^2 L^2)`: với `n = 1000`, `L = 30` khoảng `9 * 10^8` trong tệ nhất. Space `O(1)` thêm.

**Nút thắt:** tra cứu trong **list** quá chậm. Hãy đếm số lần so sánh chuỗi trong trường hợp xấu.

In [9]:
def brute_comparisons(words):
    cnt = 0
    for w in words:
        for k in range(1, len(w)):
            cnt += len(words)               # mỗi lần tra list ~ n phép so sánh
            if w[:k] not in words:
                break
    return cnt

rng = random.Random(1)
words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 30))) for _ in range(1000)]
print("số phép so sánh chuỗi ước lượng:", brute_comparisons(words))
chain = ["a" * k for k in range(1, 31)] * 33
print("chuỗi 'a'*k (mọi tiền tố đều có):", brute_comparisons(chain))

số phép so sánh chuỗi ước lượng: 5034000
chuỗi 'a'*k (mọi tiền tố đều có): 14211450


## 6. Ý tưởng thuật toán / bước đột phá

Hai cải tiến độc lập:

1. **Đổi list thành set**: tra tiền tố trong `O(1)` (băm) thay vì `O(n)`.
2. **Dùng tính đệ quy "từ cha"**: `w` xây được ⇔ `w[:-1]` xây được. Nếu duyệt **từ ngắn đến dài** (`sorted` theo `(len, w)`) và giữ tập `built` các từ đã xây được (khởi tạo `{""}` để từ 1 ký tự có cha hợp lệ), thì với mỗi từ chỉ cần **một lần tra** `w[:-1] in built`, thay vì kiểm tra `len(w) - 1` tiền tố.

Ngoài ra, khi duyệt theo `(len, w)` tăng dần, từ **đầu tiên** đạt độ dài mới đã là từ nhỏ nhất theo từ điển trong độ dài đó; do đó chỉ cần cập nhật `ans` khi **dài hơn hẳn**.

In [10]:
def demo_built(words):
    built = {""}
    for w in sorted(set(words), key=lambda x: (len(x), x)):
        parent_ok = w[:-1] in built
        if parent_ok:
            built.add(w)
        print(f"{w!r:9} cha {w[:-1]!r:7} trong built? {parent_ok!s:5} -> built = {sorted(built - {''})}")

demo_built(["a", "banana", "app", "appl", "ap", "apply", "apple"])

'a'       cha ''      trong built? True  -> built = ['a']
'ap'      cha 'a'     trong built? True  -> built = ['a', 'ap']
'app'     cha 'ap'    trong built? True  -> built = ['a', 'ap', 'app']
'appl'    cha 'app'   trong built? True  -> built = ['a', 'ap', 'app', 'appl']
'apple'   cha 'appl'  trong built? True  -> built = ['a', 'ap', 'app', 'appl', 'apple']
'apply'   cha 'appl'  trong built? True  -> built = ['a', 'ap', 'app', 'appl', 'apple', 'apply']
'banana'  cha 'banan' trong built? False -> built = ['a', 'ap', 'app', 'appl', 'apple', 'apply']


## 7. Các cách tiếp theo

Phần này có thêm 4 cách (tổng 5 cách): **set các từ + kiểm tra mọi tiền tố**, **sắp xếp + `built`** (lời giải nộp), **Trie + tra từng từ**, **Trie + DFS**.

### 7.1. Cách 2 — Tập băm các từ, kiểm tra mọi tiền tố

**Giải thích từng đoạn.**

1. `s = set(words)`.
2. Với mỗi `w`, kiểm tra `all(w[:k] in s for k in range(1, len(w)))`.
3. Cập nhật đáp án theo quy tắc: dài hơn, hoặc cùng độ dài mà nhỏ hơn theo từ điển.

In [11]:
def longest_set(words):
    s = set(words)
    ans = ""
    for w in words:
        if all(w[:k] in s for k in range(1, len(w))):
            if len(w) > len(ans) or (len(w) == len(ans) and w < ans):
                ans = w
    return ans

for words, expected in examples:
    got = longest_set(words)
    print(repr(got))
    assert got == expected

'world'
'apple'


In [12]:
class Solution:
    def longestWord(self, words: List[str]) -> str:
        s = set(words)
        ans = ""
        for w in words:
            if all(w[:k] in s for k in range(1, len(w))):
                if len(w) > len(ans) or (len(w) == len(ans) and w < ans):
                    ans = w
        return ans

print(repr(Solution().longestWord(["w", "wo", "wor", "worl", "world"])))

'world'


**Độ phức tạp:** mỗi từ kiểm tra tối đa `L` tiền tố, mỗi lần cắt + băm tốn `O(L)` → Time `O(n * L^2)` (khoảng `10^6` với giới hạn đề), Space `O(n * L)` cho tập.

### 7.2. Cách 3 — Sắp xếp + tập `built` (lời giải nộp)

**Giải thích từng đoạn.**

1. `built = {""}`: coi chuỗi rỗng là "đã xây được" để từ 1 ký tự hợp lệ ngay.
2. Duyệt `sorted(words, key=lambda w: (len(w), w))` — ngắn trước dài sau, cùng độ dài thì từ điển.
3. Với `w`: nếu `w[:-1] in built` thì `w` xây được → `built.add(w)`, và nếu `len(w) > len(ans)` thì `ans = w`.
4. Chỉ cập nhật khi **dài hơn hẳn** (từ đầu tiên của mỗi độ dài đã nhỏ nhất theo từ điển).
5. Từ trùng nhau không gây vấn đề: lần thứ hai `w[:-1]` vẫn trong `built`, `len(w) > len(ans)` sai nên không đổi.

In [13]:
def longest_sorted(words):
    built = {""}
    ans = ""
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[:-1] in built:
            built.add(w)
            if len(w) > len(ans):
                ans = w
    return ans

for words, expected in examples:
    got = longest_sorted(words)
    print(repr(got))
    assert got == expected
print(repr(longest_sorted(["b", "ab"])), repr(longest_sorted(["x", "x"])))

'world'
'apple'
'b' 'x'


In [14]:
class Solution:
    def longestWord(self, words: List[str]) -> str:
        built = {""}
        ans = ""
        for w in sorted(words, key=lambda x: (len(x), x)):
            if w[:-1] in built:
                built.add(w)
                if len(w) > len(ans):
                    ans = w
        return ans

print(repr(Solution().longestWord(["a", "banana", "app", "appl", "ap", "apply", "apple"])))

'apple'


**Độ phức tạp:** sắp xếp `O(n log n * L)` (so sánh chuỗi `O(L)`), vòng duyệt `O(n * L)` (cắt `w[:-1]` và băm). Tổng Time `O(n L log n)`, Space `O(n L)`. Đây là cách ngắn gọn và nhanh nhất để viết.

### 7.3. Cách 4 — Trie + tra từng từ

**Giải thích từng đoạn.**

1. Chèn mọi từ vào Trie (`is_end = True` ở nút cuối mỗi từ).
2. `search(w)`: đi theo từng ký tự; **nếu nút nào dọc đường không phải `is_end`** → `False` (một tiền tố không phải từ). Đi hết mà mọi nút đều `is_end` → `True`.
3. Duyệt mọi từ, cập nhật theo quy tắc dài/từ điển.

In [15]:
class Trie2(Trie):
    def search(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
            if not node.is_end:             # tiền tố này không phải một từ
                return False
        return True

def longest_trie_search(words):
    trie = Trie2()
    for w in words:
        trie.insert(w)
    ans = ""
    for w in words:
        if trie.search(w) and (len(w) > len(ans) or (len(w) == len(ans) and w < ans)):
            ans = w
    return ans

for words, expected in examples:
    got = longest_trie_search(words)
    print(repr(got))
    assert got == expected

'world'
'apple'


In [16]:
class Trie:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def insert(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = Trie()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
            if not node.is_end:
                return False
        return True


class Solution:
    def longestWord(self, words: List[str]) -> str:
        trie = Trie()
        for w in words:
            trie.insert(w)
        ans = ""
        for w in words:
            if trie.search(w) and (len(w) > len(ans) or (len(w) == len(ans) and w < ans)):
                ans = w
        return ans

print(repr(Solution().longestWord(["w", "wo", "wor", "worl", "world"])))

'world'


**Độ phức tạp:** chèn `O(n L)`, mỗi `search` `O(L)` → Time `O(n L)`, Space `O(n L)` (số nút tối đa). Nhanh nhất về lý thuyết (không có `log n`, không cắt chuỗi nhiều lần).

### 7.4. Cách 5 — Trie + DFS chỉ đi qua nút `is_end`

**Ý tưởng.** Từ xây được = đường đi từ gốc mà **mọi nút đều `is_end`**. Vậy ta chỉ cần DFS từ gốc, **chỉ đi vào con có `is_end = True`**, ghi nhận đường đi dài nhất. Duyệt con theo thứ tự chữ cái tăng dần và chỉ cập nhật khi **dài hơn hẳn** thì từ đầu tiên gặp ở mỗi độ sâu là nhỏ nhất theo từ điển.

In [17]:
def longest_trie_dfs(words):
    trie = Trie()
    for w in words:
        trie.insert(w)
    best = [""]

    def dfs(node, path):
        for ch in sorted(node.children):                # chữ cái tăng dần
            child = node.children[ch]
            if child.is_end:                            # chỉ đi qua các nút là từ
                new = path + ch
                if len(new) > len(best[0]):
                    best[0] = new
                dfs(child, new)

    dfs(trie, "")
    return best[0]

for words, expected in examples:
    got = longest_trie_dfs(words)
    print(repr(got))
    assert got == expected

'world'
'apple'


In [18]:
class Solution:
    def longestWord(self, words: List[str]) -> str:
        trie = Trie()
        for w in words:
            trie.insert(w)
        best = [""]

        def dfs(node, path):
            for ch in sorted(node.children):
                child = node.children[ch]
                if child.is_end:
                    new = path + ch
                    if len(new) > len(best[0]):
                        best[0] = new
                    dfs(child, new)

        dfs(trie, "")
        return best[0]

print(repr(Solution().longestWord(["a", "banana", "app", "appl", "ap", "apply", "apple"])))

'apple'


**Độ phức tạp:** Time `O(n L + số nút * 26 log 26)` ~ `O(n L)`, Space `O(n L)` cho Trie; ngăn xếp đệ quy sâu tối đa `L = 30` nên an toàn. Ưu điểm: không phải kiểm tra từng từ — chỉ duyệt phần cây hợp lệ.

## 8. TRACE từng bước

### 8.1. Trace cách 3 (sort + `built`) với ví dụ 2

Sắp xếp theo `(len, w)`: `a, ap, app, appl, apple, apply, banana`. `built = {""}`, `ans = ""`.

| từ `w` | cha `w[:-1]` | cha ∈ built? | hành động | `ans` |
|---|---|---|---|---|
| `a` | `""` | có | thêm `a`; `len 1 > 0` | `a` |
| `ap` | `a` | có | thêm `ap`; `2 > 1` | `ap` |
| `app` | `ap` | có | thêm; `3 > 2` | `app` |
| `appl` | `app` | có | thêm; `4 > 3` | `appl` |
| `apple` | `appl` | có | thêm; `5 > 4` | `apple` |
| `apply` | `appl` | có | thêm; `5 > 5` sai | `apple` |
| `banana` | `banan` | **không** | bỏ qua | `apple` |

Đáp án `apple`.

In [19]:
def trace_sorted(words):
    built = {""}
    ans = ""
    order = sorted(words, key=lambda x: (len(x), x))
    print("thứ tự duyệt:", order)
    for w in order:
        ok = w[:-1] in built
        action = "bỏ qua (cha chưa xây được)"
        if ok:
            built.add(w)
            if len(w) > len(ans):
                ans = w
                action = f"thêm, ans = {ans!r}"
            else:
                action = f"thêm, nhưng không dài hơn {ans!r}"
        print(f"{w!r:9} cha={w[:-1]!r:8} cha trong built? {ok!s:5} -> {action}")
    return ans

assert trace_sorted(["a", "banana", "app", "appl", "ap", "apply", "apple"]) == "apple"
print()
assert trace_sorted(["b", "ab", "abc", "x", "xy"]) == "xy"

thứ tự duyệt: ['a', 'ap', 'app', 'appl', 'apple', 'apply', 'banana']
'a'       cha=''       cha trong built? True  -> thêm, ans = 'a'
'ap'      cha='a'      cha trong built? True  -> thêm, ans = 'ap'
'app'     cha='ap'     cha trong built? True  -> thêm, ans = 'app'
'appl'    cha='app'    cha trong built? True  -> thêm, ans = 'appl'
'apple'   cha='appl'   cha trong built? True  -> thêm, ans = 'apple'
'apply'   cha='appl'   cha trong built? True  -> thêm, nhưng không dài hơn 'apple'
'banana'  cha='banan'  cha trong built? False -> bỏ qua (cha chưa xây được)

thứ tự duyệt: ['b', 'x', 'ab', 'xy', 'abc']
'b'       cha=''       cha trong built? True  -> thêm, ans = 'b'
'x'       cha=''       cha trong built? True  -> thêm, nhưng không dài hơn 'b'
'ab'      cha='a'      cha trong built? False -> bỏ qua (cha chưa xây được)
'xy'      cha='x'      cha trong built? True  -> thêm, ans = 'xy'
'abc'     cha='ab'     cha trong built? False -> bỏ qua (cha chưa xây được)


Ca thứ hai: `b` xây được, nhưng `ab` có cha `a` **không có trong từ điển** nên không xây được, kéo theo `abc` cũng không (cha `ab` chưa vào `built`). `x` rồi `xy` xây được → đáp án `xy` (độ dài 2).

### 8.2. Trace Trie + DFS với ví dụ 1 và một ca có nhiều nhánh

Trie của `w, wo, wor, worl, world`: một chuỗi dài, mọi nút đều `is_end` → DFS đi hết 5 nút. Với ca nhiều nhánh `["a","ab","ac","abc","acd","b","bcd"]`: gốc có hai con `a` (end) và `b` (end). Nhánh `a`: `ab` (end) → `abc` (end), `ac` (end) → `acd` (end). Nhánh `b`: có con `c` nhưng `bc` **không** là từ → không đi. Độ dài tối đa là `3`, đầu tiên gặp theo chữ cái là `abc`.

In [20]:
def trace_dfs(words):
    trie = Trie()
    for w in words:
        trie.insert(w)
    best = [""]

    def dfs(node, path, depth):
        for ch in sorted(node.children):
            child = node.children[ch]
            pad = "  " * depth
            if not child.is_end:
                print(f"{pad}{path + ch!r}: không phải từ -> không đi")
                continue
            new = path + ch
            tag = ""
            if len(new) > len(best[0]):
                best[0] = new
                tag = f"  <- kỷ lục mới, best = {new!r}"
            print(f"{pad}{new!r}: là từ{tag}")
            dfs(child, new, depth + 1)

    dfs(trie, "", 0)
    return best[0]

assert trace_dfs(["w", "wo", "wor", "worl", "world"]) == "world"
print()
assert trace_dfs(["a", "ab", "ac", "abc", "acd", "b", "bcd"]) == "abc"

'w': là từ  <- kỷ lục mới, best = 'w'
  'wo': là từ  <- kỷ lục mới, best = 'wo'
    'wor': là từ  <- kỷ lục mới, best = 'wor'
      'worl': là từ  <- kỷ lục mới, best = 'worl'
        'world': là từ  <- kỷ lục mới, best = 'world'

'a': là từ  <- kỷ lục mới, best = 'a'
  'ab': là từ  <- kỷ lục mới, best = 'ab'
    'abc': là từ  <- kỷ lục mới, best = 'abc'
  'ac': là từ
    'acd': là từ
'b': là từ
  'bc': không phải từ -> không đi


## 9. Code nộp LeetCode

Lời giải nộp là **sắp xếp + `built`** (Cách 3): ngắn gọn, `O(n L log n)` là dư sức với `n <= 1000`.

In [21]:
class Solution:
    def longestWord(self, words: List[str]) -> str:
        built = {""}
        ans = ""
        for w in sorted(words, key=lambda x: (len(x), x)):
            if w[:-1] in built:
                built.add(w)
                if len(w) > len(ans):
                    ans = w
        return ans


sol = Solution()
for words, expected in examples:
    got = sol.longestWord(words)
    print(words, "->", repr(got))
    assert got == expected

['w', 'wo', 'wor', 'worl', 'world'] -> 'world'
['a', 'banana', 'app', 'appl', 'ap', 'apply', 'apple'] -> 'apple'


## 10. Vì sao đúng

**Định nghĩa.** Gọi `B` là tập các từ **xây được**: `w ∈ B` ⇔ `w ∈ words` và (`len(w) == 1` hoặc `w[:-1] ∈ B`). Đây chính là điều kiện "mọi tiền tố độ dài `1..len` đều thuộc `words`", vì ta mở rộng đệ quy: `w[:-1] ∈ B` kéo theo `w[:-2] ∈ B`, v.v.

**Mệnh đề 1 (đúng thứ tự xử lý).** Trong thứ tự `(len, w)` tăng dần, cha `w[:-1]` có độ dài ngắn hơn nên **luôn được xét trước** `w`. Do đó khi tới `w`, tập `built` đã chứa **đúng** các từ xây được có độ dài `< len(w)` (quy nạp theo độ dài). Điều kiện `w[:-1] in built` (với `built` khởi tạo `{""}`) khớp chính xác định nghĩa của `B`.

**Mệnh đề 2 (chọn đáp án).** Mọi từ xây được được xét; trong các từ có cùng độ dài, từ nhỏ nhất theo từ điển được xét **đầu tiên** (vì khóa thứ hai là chính từ). Ta chỉ cập nhật khi `len(w) > len(ans)`, nên với mỗi độ dài, `ans` giữ từ đầu tiên (nhỏ nhất). Độ dài lớn nhất thắng. Nếu `B` rỗng thì `ans` giữ `""`.

**Từ trùng nhau:** lần gặp thứ hai của `w` không làm `len(w) > len(ans)` đúng, nên không ảnh hưởng.

Ta kiểm chứng định nghĩa đệ quy bằng cách so với định nghĩa "mọi tiền tố".

In [22]:
def buildable_by_all_prefixes(words):
    s = set(words)
    return {w for w in s if all(w[:k] in s for k in range(1, len(w) + 1))}

def buildable_by_sorted(words):
    built = {""}
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[:-1] in built:
            built.add(w)
    return built - {""}

rng = random.Random(10)
for _ in range(2000):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 10))]
    assert buildable_by_all_prefixes(words) == buildable_by_sorted(words), words
print("tập từ xây được: hai định nghĩa trùng nhau trên 2000 test")

tập từ xây được: hai định nghĩa trùng nhau trên 2000 test


## 11. Time / Space Complexity

Gọi `n` là số từ, `L` độ dài từ tối đa (`<= 30`).

| Cách | Time | Space |
|---|---|---|
| 1. Kiểm tra tiền tố trong list | `O(n^2 L^2)` | `O(1)` |
| 2. Set + kiểm tra mọi tiền tố | `O(n L^2)` | `O(n L)` |
| 3. Sắp xếp + `built` (nộp) | `O(n L log n)` | `O(n L)` |
| 4. Trie + tra từng từ | `O(n L)` | `O(n L)` |
| 5. Trie + DFS | `O(n L)` | `O(n L)` |

Với đề: `n L = 3 * 10^4` nên mọi cách trừ cách 1 đều chạy tức thì. Ta đếm số nút Trie và độ sâu DFS để thấy giới hạn.

In [23]:
def count_nodes(words):
    trie = Trie()
    nodes = 0
    for w in words:
        node = trie
        for ch in w:
            if ch not in node.children:
                node.children[ch] = Trie()
                nodes += 1
            node = node.children[ch]
        node.is_end = True
    return nodes

rng = random.Random(11)
words = ["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(30)) for _ in range(1000)]
print("1000 từ ngẫu nhiên dài 30: số nút <=", 1000 * 30, "thực tế", count_nodes(words))
words = ["a" * k for k in range(1, 31)] * 33
print("chuỗi tiền tố chung: số nút", count_nodes(words), "(chỉ 30 nút vì mọi từ chung đường)")

1000 từ ngẫu nhiên dài 30: số nút <= 30000 thực tế 28517
chuỗi tiền tố chung: số nút 30 (chỉ 30 nút vì mọi từ chung đường)


## 12. Bộ test

Gồm: ví dụ đề, biên (một từ, không đáp án, từ trùng nhau, hòa độ dài), phản ví dụ cho lời giải sai, đối chiếu 5 cách bằng `assert` với **oracle độc lập** (lặp tới điểm bất động: không phụ thuộc vào thứ tự hay tiền tố), test ngẫu nhiên có seed đúng ràng buộc, và test kích thước lớn.

In [24]:
def oracle_fixpoint(words):
    """Oracle độc lập: lặp đến khi không thêm được từ nào. Không dùng sorted/prefix-all/Trie."""
    pool = set(words)
    built = set()
    changed = True
    while changed:
        changed = False
        for w in pool:
            if w not in built and (len(w) == 1 or w[:-1] in built):
                built.add(w)
                changed = True
    best = ""
    for w in built:
        if len(w) > len(best) or (len(w) == len(best) and w < best):
            best = w
    return best

ALL_METHODS = {
    "brute": longest_brute,
    "set": longest_set,
    "sorted": longest_sorted,
    "trie_search": longest_trie_search,
    "trie_dfs": longest_trie_dfs,
    "leetcode": Solution().longestWord,
}

fixed_cases = [
    (["w", "wo", "wor", "worl", "world"], "world"),
    (["a", "banana", "app", "appl", "ap", "apply", "apple"], "apple"),
    (["a"], "a"),
    (["abc"], ""),                                   # không có từ 1 ký tự
    (["b", "ab"], "b"),                              # 'ab' cần 'a' nhưng 'a' không có
    (["a", "b", "c"], "a"),                          # hòa: nhỏ nhất theo từ điển
    (["a", "a", "a"], "a"),                          # trùng nhau
    (["yo", "ew", "fc", "zrc", "yodn", "fcm", "qm", "qmo", "fcmz", "z", "ewq", "yod", "ewqz", "y"], "yodn"),
    (["m", "mo", "moc", "moch", "mocha", "l", "la", "lat", "latt", "latte", "c", "ca", "cat"], "latte"),
    (["abc", "ab", "a"], "abc"),                     # thứ tự đầu vào ngược
    (["ab", "abc", "a"], "abc"),
    (["a", "ac", "abc"], "ac"),                      # abc cần 'ab' (không có)
    (["z", "zz", "zzz", "a", "aa", "aaa"], "aaa"),   # hòa độ dài 3
]
for words, expected in fixed_cases:
    assert oracle_fixpoint(words) == expected, (words, oracle_fixpoint(words), expected)
    for name, fn in ALL_METHODS.items():
        assert fn(list(words)) == expected, (name, words, fn(list(words)), expected)
print("test cố định:", len(fixed_cases), "bộ x", len(ALL_METHODS), "cách: OK")

test cố định: 13 bộ x 6 cách: OK


In [25]:
# Phản ví dụ 1: chỉ kiểm tra "cha có trong words" mà KHÔNG kiểm tra cha xây được
def wrong_parent_only(words):
    s = set(words)
    ans = ""
    for w in words:
        if len(w) == 1 or w[:-1] in s:
            if len(w) > len(ans) or (len(w) == len(ans) and w < ans):
                ans = w
    return ans

for words in [["b", "abc", "ab"], ["x", "abcd", "abc"]]:
    print(words, "| sai:", repr(wrong_parent_only(words)), "| đúng:", repr(oracle_fixpoint(words)))

# Phản ví dụ 2: quên quy tắc hòa độ dài (lấy từ gặp sau/ trước thay vì từ điển nhỏ nhất)
def wrong_tie(words):
    ans = ""
    s = set(words)
    for w in words:
        if all(w[:k] in s for k in range(1, len(w))) and len(w) >= len(ans):   # SAI: >= ghi đè bằng từ gặp sau
            ans = w
    return ans

print(repr(wrong_tie(["a", "b", "c"])), "| đúng:", repr(oracle_fixpoint(["a", "b", "c"])))
print(repr(wrong_tie(["apple", "apply", "a", "ap", "app", "appl"])), "| đúng:", repr(oracle_fixpoint(["apple", "apply", "a", "ap", "app", "appl"])))

['b', 'abc', 'ab'] | sai: 'abc' | đúng: 'b'
['x', 'abcd', 'abc'] | sai: 'abcd' | đúng: 'x'
'c' | đúng: 'a'
'apply' | đúng: 'apple'


In [26]:
# Đối chiếu mọi cách bằng oracle trên rất nhiều input ngẫu nhiên nhỏ (bảng chữ cái nhỏ => nhiều chuỗi lồng nhau)
rng = random.Random(720)
for _ in range(5000):
    alphabet = rng.choice(["a", "ab", "abc"])
    n = rng.randint(1, 12)
    words = ["".join(rng.choice(alphabet) for _ in range(rng.randint(1, 5))) for _ in range(n)]
    ref = oracle_fixpoint(words)
    for name, fn in ALL_METHODS.items():
        assert fn(list(words)) == ref, (name, words, fn(list(words)), ref)
print("5000 test ngẫu nhiên nhỏ: mọi cách khớp oracle")

# Vét cạn: MỌI tập từ gồm tối đa 3 từ trên {a,b}, độ dài <= 3
from itertools import product, combinations
pool = ["".join(t) for n in range(1, 4) for t in product("ab", repeat=n)]      # 14 từ
cnt = 0
for r in range(1, 4):
    for combo in combinations(pool, r):
        words = list(combo)
        ref = oracle_fixpoint(words)
        for name, fn in ALL_METHODS.items():
            assert fn(list(words)) == ref, (name, words)
        cnt += 1
print("vét cạn", cnt, "tập từ nhỏ: OK")

5000 test ngẫu nhiên nhỏ: mọi cách khớp oracle


vét cạn 469 tập từ nhỏ: OK


In [27]:
# Test lớn đúng ràng buộc: 1000 từ, độ dài <= 30, sinh bằng cách "mọc" từ các từ có sẵn để có nhiều từ xây được
rng = random.Random(7200)

def grow_words(rng, n, alphabet, max_len, p_extend):
    words = []
    while len(words) < n:
        if words and rng.random() < p_extend:
            base = rng.choice(words)
            if len(base) < max_len:
                words.append(base + rng.choice(alphabet))
                continue
        words.append("".join(rng.choice(alphabet) for _ in range(rng.randint(1, 3))))
    return words

for alphabet, p in [("ab", 0.9), ("abcdefghijklmnopqrstuvwxyz", 0.95), ("a", 1.0)]:
    words = grow_words(rng, 1000, alphabet, 30, p)
    assert len(words) == 1000 and all(1 <= len(w) <= 30 for w in words)
    ref = oracle_fixpoint(words)
    for name in ["set", "sorted", "trie_search", "trie_dfs", "leetcode"]:
        assert ALL_METHODS[name](list(words)) == ref, name
    print(f"alphabet {len(alphabet)} chữ: đáp án dài {len(ref)}: {ref[:20]!r}...  OK")

# Biên: chuỗi dài nhất có thể và không có đáp án
chain = ["a" * k for k in range(1, 31)]
assert Solution().longestWord(chain) == "a" * 30
assert Solution().longestWord(["b" * k for k in range(2, 31)]) == ""
print("biên OK")

alphabet 2 chữ: đáp án dài 16: 'bbabaaababbaaaaa'...  OK
alphabet 26 chữ: đáp án dài 10: 'tofsbumsjl'...  OK
alphabet 1 chữ: đáp án dài 0: ''...  OK
biên OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra cha **có trong `words`** mà không kiểm tra cha **xây được**

Điều kiện `w[:-1] in words` chưa đủ: cha có thể có trong từ điển nhưng bản thân cha **không xây được** (thiếu tiền tố ngắn hơn). Ví dụ `["b", "abc", "ab"]`: `abc` có cha `ab` trong từ điển, nhưng `ab` có cha `a` **không** có → `abc` không xây được. Đã thấy ở mục 12 (`wrong_parent_only`): trả `abc` thay vì `b`.

Sửa: dùng tập `built` các từ **đã được xác nhận xây được**, hoặc kiểm tra mọi tiền tố, hoặc Trie với `is_end` dọc đường.

### Lỗi 2 — Quên quy tắc hòa độ dài

Nếu dùng `len(w) >= len(ans)` thì từ gặp **sau** ghi đè từ gặp trước, kết quả phụ thuộc thứ tự đầu vào chứ không phải thứ tự từ điển. `wrong_tie(["a","b","c"])` trả `"c"`, đáp án đúng là `"a"`. Sửa: duyệt theo `(len, w)` và cập nhật khi dài hơn hẳn, hoặc so sánh từ điển rõ ràng khi hòa.

### Lỗi 3 — Quên trả `""` khi không có đáp án

Nếu khởi tạo `ans = words[0]` thay vì `""`, các ca như `["abc"]` (không có từ 1 ký tự) sẽ trả `"abc"` sai. Khởi tạo `ans = ""`.

In [28]:
def wrong_init(words):
    built = {""}
    ans = words[0]                         # SAI: nên là ""
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[:-1] in built:
            built.add(w)
            if len(w) > len(ans):
                ans = w
    return ans

for words in [["abc"], ["xy", "z"]]:
    print(words, "| sai:", repr(wrong_init(words)), "| đúng:", repr(Solution().longestWord(words)))

['abc'] | sai: 'abc' | đúng: ''
['xy', 'z'] | sai: 'xy' | đúng: 'z'


### Lỗi 4 — Quên khởi tạo `built = {""}` (từ 1 ký tự bị loại)

Với `built = set()`, mọi từ 1 ký tự có cha `""` **không** nằm trong `built`, nên không có từ nào xây được và đáp án luôn `""`. Hoặc phải xử lý riêng `len(w) == 1`.

In [29]:
def wrong_empty_built(words):
    built = set()                          # SAI: thiếu ""
    ans = ""
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[:-1] in built:
            built.add(w)
            if len(w) > len(ans):
                ans = w
    return ans

print(repr(wrong_empty_built(["w", "wo", "wor"])), "| đúng:", repr(Solution().longestWord(["w", "wo", "wor"])))

'' | đúng: 'wor'


### Lỗi 5 — Nhầm tiền tố với chuỗi con (hoặc hậu tố)

Điều kiện đề là mỗi bước thêm ký tự vào **cuối**; vì vậy các từ trung gian là **tiền tố**. Nếu kiểm tra `w[1:] in s` (bỏ ký tự **đầu**) hoặc "chứa" bất kỳ chuỗi con nào, ta giải một bài khác (các từ xây bằng cách thêm vào đầu). Hãy luôn dùng `w[:-1]`.

In [30]:
def wrong_suffix_direction(words):
    built = {""}
    ans = ""
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[1:] in built:                 # SAI: bỏ ký tự đầu (thêm ở phía trước)
            built.add(w)
            if len(w) > len(ans):
                ans = w
    return ans

words = ["a", "ba", "cba", "ab", "abc"]
print("w[1:] (thêm vào đầu):", repr(wrong_suffix_direction(words)), "| đề yêu cầu (thêm vào cuối):", repr(Solution().longestWord(words)))

w[1:] (thêm vào đầu): 'cba' | đề yêu cầu (thêm vào cuối): 'abc'


## 14. Mẹo & nhận dạng pattern

- **"Từ xây từng ký tự từ các từ khác trong từ điển"** → quan hệ cha-con `w ↔ w[:-1]`: thường giải bằng **sắp xếp theo độ dài + tập đã xây** hoặc **Trie với `is_end`**.
- **Sắp xếp với khóa `(len, w)`** vừa đảm bảo cha đứng trước con vừa xử lý tie-break từ điển "miễn phí".
- Khi cần "**chuỗi rỗng làm gốc**", khởi tạo `{""}` để bớt một nhánh `if len(w) == 1`.
- Trie: từ hợp lệ khi **mọi nút trên đường đi đều là từ** (`is_end`). DFS trên Trie chỉ đi qua nút `is_end` cho kết quả trực tiếp, không cần hỏi lại từng từ.
- Tie-break "dài nhất, hòa thì nhỏ nhất theo từ điển": hoặc dùng `min(cands, key=lambda w: (-len(w), w))`, hoặc duyệt theo thứ tự sao cho từ đầu tiên của mỗi độ dài đã là nhỏ nhất.
- Họ bài liên quan: *Longest String Chain* (1048), *Replace Words* (648), *Implement Trie* (208), *Longest Word in Dictionary through Deleting* (524).

## 15. Biến thể

### 15.1. Trả về **tất cả** các từ xây được dài nhất

Thay vì chỉ một từ, trả mọi từ có độ dài tối đa (đã loại trùng, sắp xếp từ điển).

### 15.2. Đếm số từ xây được

Đếm các từ **khác nhau** có thể xây được.

### 15.3. Thêm ký tự vào **đầu** thay vì cuối

Dùng `w[1:]` làm "cha" (hoặc đảo ngược mọi từ rồi dùng lại thuật toán).

### 15.4. Chuỗi dài nhất khi mỗi bước chèn **một ký tự ở vị trí bất kỳ** (LeetCode 1048)

Đây là DP thực sự (cha không còn duy nhất); ta chỉ nhắc đến, không cài đặt.

In [31]:
def all_longest(words):
    """15.1: mọi từ xây được có độ dài lớn nhất (đã loại trùng và sắp xếp)."""
    built = {""}
    for w in sorted(set(words), key=lambda x: (len(x), x)):
        if w[:-1] in built:
            built.add(w)
    built.discard("")
    if not built:
        return []
    m = max(map(len, built))
    return sorted(w for w in built if len(w) == m)

def count_buildable(words):
    """15.2: số từ KHÁC NHAU xây được."""
    built = {""}
    for w in sorted(set(words), key=lambda x: (len(x), x)):
        if w[:-1] in built:
            built.add(w)
    return len(built) - 1

def longest_front(words):
    """15.3: mỗi bước thêm ký tự vào ĐẦU; cha là w[1:]."""
    built = {""}
    ans = ""
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[1:] in built:
            built.add(w)
            if len(w) > len(ans):
                ans = w
    return ans

print(all_longest(["a", "banana", "app", "appl", "ap", "apply", "apple"]))
print(count_buildable(["a", "banana", "app", "appl", "ap", "apply", "apple"]))
print(longest_front(["a", "ba", "cba", "ab", "abc"]))

# kiểm chứng bằng oracle độc lập
rng = random.Random(15)
for _ in range(1000):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 10))]
    b = buildable_by_all_prefixes(words)
    assert count_buildable(words) == len(b)
    exp_all = sorted(w for w in b if len(w) == max(map(len, b))) if b else []
    assert all_longest(words) == exp_all
print("các biến thể 15.1-15.3 OK")

['apple', 'apply']
6
cba
các biến thể 15.1-15.3 OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tiền tố trong list | Với mỗi từ tra từng tiền tố trong list | `O(n^2 L^2)` | `O(1)` | Chỉ để hiểu đề |
| 2. Set + mọi tiền tố | Tra tiền tố bằng băm | `O(n L^2)` | `O(n L)` | Gọn, đủ nhanh với đề |
| 3. Sắp xếp + `built` | Ngắn→dài, chỉ tra cha | `O(n L log n)` | `O(n L)` | **Nộp bài**, ngắn nhất, tie-break miễn phí |
| 4. Trie + tra từng từ | `is_end` dọc đường | `O(n L)` | `O(n L)` | Muốn `O(n L)` và cấu trúc Trie tái dùng |
| 5. Trie + DFS | Chỉ đi qua nút `is_end` | `O(n L)` | `O(n L)` | Muốn duyệt cây trực tiếp, không hỏi từng từ |

In [32]:
rng = random.Random(16)
words = grow_words(rng, 1000, "abc", 30, 0.9)
answers = {name: fn(list(words)) for name, fn in ALL_METHODS.items() if name != "brute"}
print(answers)
assert len(set(answers.values())) == 1

{'set': 'bbbacabcaabcaaa', 'sorted': 'bbbacabcaabcaaa', 'trie_search': 'bbbacabcaabcaaa', 'trie_dfs': 'bbbacabcaabcaaa', 'leetcode': 'bbbacabcaabcaaa'}


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(sorted(["ab", "b", "a", "abc"], key=lambda w: (len(w), w)))
print("apple" < "apply", "app" < "ap")
print(Solution().longestWord(["a", "ab", "b", "bc", "bcd"]))
```

In [33]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `words = ["t", "ti", "tin", "tint", "to", "toy", "x", "xyz"]`, hãy liệt kê các từ **xây được** và cho biết đáp án. Vì sao `xyz` bị loại?

In [34]:
# Bài 2: điền kết quả tính tay
hand_buildable = None      # list các từ xây được (sắp xếp theo (độ dài, từ điển))
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành phiên bản sort + `built`.

In [35]:
def exercise_3_longest(words: List[str]) -> str:
    built = {""}
    ans = ""
    # TODO 1: duyệt words đã sắp xếp theo khóa (len(w), w)
    #     TODO 2: nếu "cha" w[:-1] nằm trong built
    #         TODO 3: thêm w vào built
    #         TODO 4: nếu w dài hơn HẲN ans thì cập nhật ans
    return ans


# assert exercise_3_longest(["w", "wo", "wor", "worl", "world"]) == "world"
# assert exercise_3_longest(["a", "banana", "app", "appl", "ap", "apply", "apple"]) == "apple"
# assert exercise_3_longest(["abc"]) == ""

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 4, hãy tự viết `exercise_4_trie(words)`: xây Trie, rồi với mỗi từ kiểm tra mọi nút dọc đường đều `is_end`; chọn dài nhất / nhỏ nhất từ điển.

In [36]:
def exercise_4_trie(words: List[str]) -> str:
    # TODO
    pass


# assert exercise_4_trie(["w", "wo", "wor", "worl", "world"]) == "world"
# assert exercise_4_trie(["b", "ab"]) == "b"
# assert exercise_4_trie(["a", "b", "c"]) == "a"

### Bài 5 — Mini challenge: thêm ký tự vào đầu hoặc cuối (khó)

Giống đề gốc nhưng mỗi bước được thêm một ký tự vào **đầu hoặc cuối** của từ trước đó (từ 1 ký tự luôn là khởi đầu hợp lệ nếu có trong từ điển). Viết `exercise_5_either(words)` trả về từ dài nhất xây được (hòa: nhỏ nhất từ điển; không có: `""`). Gợi ý: một từ `w` (độ dài `>= 2`) xây được khi `w[:-1]` **hoặc** `w[1:]` xây được.

```python
exercise_5_either(["a", "ba", "cba"])        -> "cba"
exercise_5_either(["a", "ab", "xab", "zb"])  -> "xab"
exercise_5_either(["ab", "abc"])             -> ""
```

In [37]:
def exercise_5_either(words: List[str]) -> str:
    # TODO
    pass


# assert exercise_5_either(["a", "ba", "cba"]) == "cba"
# assert exercise_5_either(["a", "ab", "xab", "zb"]) == "xab"
# assert exercise_5_either(["ab", "abc"]) == ""

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `sorted` với khóa `(len, w)` xếp ngắn trước, hòa thì từ điển.
- Mức 2: `"app" < "ap"` là `False` vì `"ap"` là tiền tố của `"app"` nên nhỏ hơn.
- Mức 3: dòng 1 `['a', 'b', 'ab', 'abc']`; dòng 2 `True False`; dòng 3: `a, ab` xây được (độ dài 2), `b, bc, bcd` xây được (độ dài 3) → `bcd`.

**Bài 2**
- Mức 1: chạy qua từng từ, kiểm tra tiền tố.
- Mức 2: `t, ti, tin, tint, to, toy` đều xây được; `x` xây được.
- Mức 3: `xyz` có cha `xy` không có trong từ điển. Từ dài nhất: `tint` (4). Xây được: `['t', 'x', 'ti', 'to', 'tin', 'toy', 'tint']`; đáp án `tint`.

**Bài 3**
- Mức 1: giống mục 9.
- Mức 2: `for w in sorted(words, key=lambda x: (len(x), x))`.
- Mức 3: `if w[:-1] in built: built.add(w); if len(w) > len(ans): ans = w`.

**Bài 4**
- Mức 1: dùng lớp `Trie` ở mục 7.3.
- Mức 2: `search(w)`: đi theo ký tự, nếu nút nào không `is_end` thì `False`.
- Mức 3: duyệt mọi `w`, cập nhật khi `len(w) > len(ans)` hoặc (`==` và `w < ans`).

**Bài 5**
- Mức 1: giữ `built = set()`; từ 1 ký tự xây được ngay (nếu có trong từ điển).
- Mức 2: duyệt theo độ dài tăng dần; với `len(w) >= 2`, xây được nếu `w[:-1] in built or w[1:] in built`.
- Mức 3: đặt `built = {""}` thì `w[:-1]` và `w[1:]` của từ 1 ký tự đều là `""`; cập nhật `ans` khi dài hơn hẳn (khóa `(len, w)` đã lo tie-break).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(sorted(["ab", "b", "a", "abc"], key=lambda w: (len(w), w)))
print("apple" < "apply", "app" < "ap")
print(Solution().longestWord(["a", "ab", "b", "bc", "bcd"]))

assert sorted(["ab", "b", "a", "abc"], key=lambda w: (len(w), w)) == ["a", "b", "ab", "abc"]
assert ("apple" < "apply", "app" < "ap") == (True, False)
assert Solution().longestWord(["a", "ab", "b", "bc", "bcd"]) == "bcd"

['a', 'b', 'ab', 'abc']
True False
bcd


### Đáp án Bài 2

In [39]:
words = ["t", "ti", "tin", "tint", "to", "toy", "x", "xyz"]
b = sorted(buildable_by_all_prefixes(words), key=lambda w: (len(w), w))
print(b, "| đáp án:", Solution().longestWord(words))
assert b == ["t", "x", "ti", "to", "tin", "toy", "tint"]
assert Solution().longestWord(words) == "tint"
assert "xy" not in words            # nên "xyz" bị loại

['t', 'x', 'ti', 'to', 'tin', 'toy', 'tint'] | đáp án: tint


### Đáp án Bài 3

In [40]:
def exercise_3_solution(words: List[str]) -> str:
    built = {""}
    ans = ""
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[:-1] in built:
            built.add(w)
            if len(w) > len(ans):
                ans = w
    return ans

assert exercise_3_solution(["w", "wo", "wor", "worl", "world"]) == "world"
assert exercise_3_solution(["a", "banana", "app", "appl", "ap", "apply", "apple"]) == "apple"
assert exercise_3_solution(["abc"]) == ""
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_solution(words: List[str]) -> str:
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True                         # cờ is_end

    def ok(w):
        node = root
        for ch in w:
            node = node[ch]
            if "$" not in node:
                return False
        return True

    ans = ""
    for w in words:
        if ok(w) and (len(w) > len(ans) or (len(w) == len(ans) and w < ans)):
            ans = w
    return ans

assert exercise_4_solution(["w", "wo", "wor", "worl", "world"]) == "world"
assert exercise_4_solution(["b", "ab"]) == "b"
assert exercise_4_solution(["a", "b", "c"]) == "a"
rng = random.Random(44)
for _ in range(500):
    ws = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 10))]
    assert exercise_4_solution(ws) == oracle_fixpoint(ws), ws
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_solution(words: List[str]) -> str:
    built = {""}
    ans = ""
    for w in sorted(words, key=lambda x: (len(x), x)):
        if w[:-1] in built or w[1:] in built:
            built.add(w)
            if len(w) > len(ans):
                ans = w
    return ans

assert exercise_5_solution(["a", "ba", "cba"]) == "cba"
assert exercise_5_solution(["a", "ab", "xab", "zb"]) == "xab"
assert exercise_5_solution(["ab", "abc"]) == ""

# oracle độc lập: lặp tới điểm bất động
def either_oracle(words):
    pool = set(words)
    built = set()
    changed = True
    while changed:
        changed = False
        for w in pool:
            if w not in built and (len(w) == 1 or w[:-1] in built or w[1:] in built):
                built.add(w)
                changed = True
    best = ""
    for w in built:
        if len(w) > len(best) or (len(w) == len(best) and w < best):
            best = w
    return best

rng = random.Random(55)
for _ in range(1000):
    ws = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 10))]
    assert exercise_5_solution(ws) == either_oracle(ws), ws
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template sắp xếp + tập đã xây

```python
class Solution:
    def longestWord(self, words: List[str]) -> str:
        built = {""}                                   # chuỗi rỗng là gốc
        ans = ""
        for w in sorted(words, key=lambda x: (len(x), x)):   # ngắn -> dài, hòa: từ điển
            if w[:-1] in built:                        # cha đã xây được
                built.add(w)
                if len(w) > len(ans):                  # chỉ khi dài hơn HẲN
                    ans = w
        return ans
```

Câu thần chú: **"ngắn trước dài sau; có cha xây được thì mình xây được; chỉ đổi đáp án khi dài hơn hẳn."**

### Checklist trước khi submit

- [ ] Điều kiện xây được là **cha xây được** (không chỉ "cha có trong từ điển"), hoặc mọi tiền tố có trong từ điển.
- [ ] `built = {""}` để từ 1 ký tự hợp lệ; `ans = ""` để trả chuỗi rỗng khi không có đáp án.
- [ ] Hòa độ dài → nhỏ nhất theo từ điển (sắp xếp `(len, w)` + cập nhật khi dài hơn hẳn).
- [ ] Từ trùng nhau không làm hỏng kết quả.
- [ ] Test: `["abc"]` (không đáp án), `["b", "ab"]`, `["a","b","c"]` (hòa), thứ tự đầu vào đảo ngược.
- [ ] Skeleton đúng: `class Solution`, method `longestWord(self, words: List[str]) -> str`.